# Task 4: General Health Query Chatbot (Prompt Engineering)

**DevelopersHub Corporation - AI/ML Engineering Internship**

## Objective
Create a chatbot that answers general health-related questions using an LLM. Uses prompt engineering to ensure responses are friendly, clear, and include appropriate safety disclaimers.

## 1. Import Required Libraries

In [ ]:
import requests
import json
import re
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print('Libraries imported successfully')

## 2. Configuration

Choose between:
- **Hugging Face Inference API** (free, no API key needed for some models)
- **OpenAI API** (requires API key)

We'll use Hugging Face's free Mistral-7B-Instruct model as the default.

In [ ]:
# Configuration
import os
from dotenv import load_dotenv
load_dotenv()

USE_HUGGINGFACE = True  # Set to False to use OpenAI

# Hugging Face settings
HF_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
HF_API_URL = f"https://api-inference.huggingface.co/models/{HF_MODEL}"

# OpenAI settings (if using)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = "gpt-3.5-turbo"

print(f"Using: {'Hugging Face (' + HF_MODEL + ')' if USE_HUGGINGFACE else 'OpenAI (' + OPENAI_MODEL + ')'}")


## 3. Safety Filters and System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a helpful and knowledgeable medical assistant AI. 
Your role is to provide general health information and education.

IMPORTANT RULES:
1. Always start with a disclaimer that you are not a replacement for professional medical advice.
2. If someone describes an emergency (chest pain, severe bleeding, difficulty breathing, etc.), 
   tell them to call emergency services immediately.
3. Do NOT prescribe medications or specific dosages.
4. Do NOT diagnose conditions - suggest that they consult a doctor.
5. Provide general wellness tips and information about common conditions.
6. Be empathetic, clear, and use simple language.
7. If unsure about something, say so rather than making up information.
8. For medication questions, always recommend consulting a doctor or pharmacist.
"""

EMERGENCY_KEYWORDS = [
    'chest pain', 'heart attack', 'stroke', 'severe bleeding', 'difficulty breathing',
    'unconscious', 'not breathing', 'suicide', 'overdose', 'severe allergic reaction',
    'head injury', 'poisoning', 'severe burn', 'drowning'
]

print('Safety filters and system prompt configured')

In [ ]:
def check_emergency(query):
    """Check if query mentions an emergency situation."""
    query_lower = query.lower()
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query_lower:
            return True
    return False

def get_emergency_response():
    """Return emergency response message."""
    return ("⚠️ **EMERGENCY DETECTED** ⚠️\n\n"
            "If you or someone else is experiencing a medical emergency, "
            "**please call emergency services immediately** (911 in US/Canada, "
            "112 in EU, 999 in UK).\n\n"
            "Do not wait for online advice - every second counts in an emergency.")

print('Safety check functions ready')

## 4. Chatbot Functions

### 4.1 Hugging Face API

In [ ]:
def query_huggingface(prompt):
    """Send query to Hugging Face Inference API."""
    headers = {"Content-Type": "application/json"}
    
    # Format prompt for Mistral instruct model
    formatted_prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\nUser question: {prompt} [/INST]"
    
    payload = {
        "inputs": formatted_prompt,
        "parameters": {
            "max_new_tokens": 500,
            "temperature": 0.7,
            "top_p": 0.95,
            "do_sample": True
        }
    }
    
    try:
        response = requests.post(HF_API_URL, headers=headers, json=payload, timeout=30)
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list) and len(result) > 0:
                return result[0].get('generated_text', '').split('[/INST]')[-1].strip()
            return str(result)
        else:
            return f"⚠️ API Error: {response.status_code}. The model may be loading. Please wait and try again."
    except Exception as e:
        return f"⚠️ Connection Error: {str(e)}. Please check your internet connection."

### 4.2 OpenAI API (Alternative)

In [ ]:
def query_openai(prompt):
    """Send query to OpenAI API."""
    if not OPENAI_API_KEY:
        return "⚠️ OpenAI API key not configured. Please add your API key in the configuration section."
    
    try:
        import openai
        openai.api_key = OPENAI_API_KEY
        
        response = openai.ChatCompletion.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=500
        )
        
        return response.choices[0].message.content
    except ImportError:
        return "⚠️ OpenAI library not installed. Run: pip install openai"
    except Exception as e:
        return f"⚠️ OpenAI Error: {str(e)}"

### 4.3 Main Chat Function

In [ ]:
def chat_with_healthbot(user_query, show_disclaimer=True):
    """Main function to get a health response."""
    
    if show_disclaimer:
        print("=" * 70)
        print("⚠️ DISCLAIMER: This chatbot provides general health information only.")
        print("It is NOT a substitute for professional medical advice, diagnosis, or treatment.")
        print("Always consult a qualified healthcare provider for medical concerns.")
        print("=" * 70)
        print()
    
    # Check for emergencies first
    if check_emergency(user_query):
        print("\n" + get_emergency_response() + "\n")
        return
    
    print(f"🧑 You: {user_query}\n")
    print("🤖 HealthBot Thinking...\n")
    
    if USE_HUGGINGFACE:
        response = query_huggingface(user_query)
    else:
        response = query_openai(user_query)
    
    print("🤖 HealthBot:")
    print(response)
    print()
    print("─" * 70)

## 5. Test the Chatbot with Example Queries

In [ ]:
# Example 1: Common symptom question
chat_with_healthbot("What causes a sore throat and how can I treat it at home?")

In [ ]:
# Example 2: Medication question
chat_with_healthbot("Is paracetamol safe for children? What dosage should I give?")

In [ ]:
# Example 3: Emergency scenario test
chat_with_healthbot("I'm having chest pain and difficulty breathing. What should I do?")

In [ ]:
# Example 4: General wellness
chat_with_healthbot("How can I improve my sleep quality naturally?")

In [ ]:
# Example 5: Diet and nutrition
chat_with_healthbot("What foods help boost the immune system?")

## 6. Interactive Chat Interface

In [ ]:
print("=" * 70)
print("💬 INTERACTIVE HEALTH CHATBOT")
print("=" * 70)
print()
print("Type your health question below (or type 'quit' to exit).")
print("⚠️ This is for general information only - not medical advice.")
print()

while True:
    user_input = input("\n🧑 You: ").strip()
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\n🤖 HealthBot: Take care! Remember to consult a doctor for medical concerns. 👋")
        break
    
    if not user_input:
        continue
    
    chat_with_healthbot(user_input, show_disclaimer=False)

## 7. Summary and Key Insights

### What was built
- A **prompt-engineered health chatbot** using either Hugging Face's free Mistral-7B or OpenAI's GPT models
- **Safety filters** to detect emergencies and redirect to professional help
- **System prompt** that enforces safe, helpful, and empathetic responses

### Prompt Engineering Techniques Used
1. **Role assignment**: "You are a helpful medical assistant"
2. **Guardrails**: Clear rules about what the AI should NOT do (diagnose, prescribe)
3. **Safety disclaimers**: Automatic disclaimer before every response
4. **Emergency detection**: Keyword-based safety check before LLM query
5. **Format instructions**: Structured response format for clarity

### Safety Features
✅ Emergency keyword detection
✅ Automatic disclaimer messages
✅ No diagnosis or prescription capabilities
✅ Clear guidance to seek professional help

### Conclusion
Prompt engineering is a powerful technique for directing LLM behavior without fine-tuning. By carefully crafting system prompts and adding safety layers, we can build helpful health information chatbots while minimizing potential harm.

In [ ]:
print("Task 4: Health Query Chatbot Complete!")